In [27]:
from openai import OpenAI

In [28]:
!pwd

/Users/anshulchiranth/Desktop/Strike/Voice Experiments


In [29]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [30]:
client = OpenAI()

In [31]:
audio_file = open("Unclipped Processed Transactions/George_8.wav", "rb")

In [32]:
transcription = client.audio.transcriptions.create(model = "gpt-4o-transcribe", file = audio_file, response_format = "text")
print(transcription)

It's a wonderful day at Dairy Queen. Will you be using your mobile rewards today? No, thank you. What can I get for you? Can I get a small pecan cluster blizzard, please? What flavor blizzard was that again? Pecan cluster, turtle pecan cluster. Yes, ma'am. And would you like to add extra pecans in there? No, thank you. What else can I get for you? That's it. That'll be $5.40 for you. Okay, thank you.



In [33]:
STEP1_SYSTEM_PROMPT = """You are a transcript processor for drive-thru audio recordings. Your job is to:

1. Add speaker labels ("Operator:" and "Customer:") to each line of the transcript
2. Extract metadata about the transaction

For each transcript, return a JSON object with these fields:

{
    "text": "The full transcript with Operator: and Customer: labels on each line",
    "complete_order": 1 if the order appears complete, 0 if garbled/incoherent/missing items/mostly non-English,
    "mobile_order": 1 if customer is picking up a mobile/online order, 0 otherwise,
    "coupon_used": 1 if a coupon was mentioned or applied, 0 otherwise,
    "asked_more_time": 1 if the operator asked the customer to wait/hold on, 0 otherwise,
    "out_of_stock_items": "comma-separated list of out-of-stock items mentioned" or "0" if none
}

Rules:
- Preserve the exact wording of the transcript
- Each line should start with either "Operator: " or "Customer: "
- If you can't determine who is speaking, make your best guess based on context
- An order is incomplete if it's mostly unintelligible, missing key items, or primarily non-English
- Mobile orders are when customers mention picking up an online/app order
- Only mark coupon_used=1 if a coupon/discount is explicitly mentioned
- asked_more_time=1 if operator says things like "hold on", "one moment", "give me a second"
"""

In [34]:
final_step1_prompt = f"""{STEP1_SYSTEM_PROMPT}

Process this transcript and return the JSON result:

---
{transcription}
---

Return ONLY the JSON object, no other text."""

In [35]:
final_step1_prompt

'You are a transcript processor for drive-thru audio recordings. Your job is to:\n\n1. Add speaker labels ("Operator:" and "Customer:") to each line of the transcript\n2. Extract metadata about the transaction\n\nFor each transcript, return a JSON object with these fields:\n\n{\n    "text": "The full transcript with Operator: and Customer: labels on each line",\n    "complete_order": 1 if the order appears complete, 0 if garbled/incoherent/missing items/mostly non-English,\n    "mobile_order": 1 if customer is picking up a mobile/online order, 0 otherwise,\n    "coupon_used": 1 if a coupon was mentioned or applied, 0 otherwise,\n    "asked_more_time": 1 if the operator asked the customer to wait/hold on, 0 otherwise,\n    "out_of_stock_items": "comma-separated list of out-of-stock items mentioned" or "0" if none\n}\n\nRules:\n- Preserve the exact wording of the transcript\n- Each line should start with either "Operator: " or "Customer: "\n- If you can\'t determine who is speaking, make

In [36]:
messages = messages = [{"role": "user", "content": final_step1_prompt}]
kwargs = {
            "model": "gpt-4o",
            "messages": messages,
            "temperature": 0,
        }

In [37]:
response = client.chat.completions.create(**kwargs)

In [38]:
response.choices[0].message.content

'```json\n{\n    "text": "Operator: It\'s a wonderful day at Dairy Queen. Will you be using your mobile rewards today?\\nCustomer: No, thank you.\\nOperator: What can I get for you?\\nCustomer: Can I get a small pecan cluster blizzard, please?\\nOperator: What flavor blizzard was that again?\\nCustomer: Pecan cluster, turtle pecan cluster.\\nOperator: Yes, ma\'am. And would you like to add extra pecans in there?\\nCustomer: No, thank you.\\nOperator: What else can I get for you?\\nCustomer: That\'s it.\\nOperator: That\'ll be $5.40 for you.\\nCustomer: Okay, thank you.",\n    "complete_order": 1,\n    "mobile_order": 0,\n    "coupon_used": 0,\n    "asked_more_time": 0,\n    "out_of_stock_items": "0"\n}\n```'

In [39]:
raw_content = response.choices[0].message.content
cleaned = raw_content.strip("```json").strip("```").strip()
cleaned

'{\n    "text": "Operator: It\'s a wonderful day at Dairy Queen. Will you be using your mobile rewards today?\\nCustomer: No, thank you.\\nOperator: What can I get for you?\\nCustomer: Can I get a small pecan cluster blizzard, please?\\nOperator: What flavor blizzard was that again?\\nCustomer: Pecan cluster, turtle pecan cluster.\\nOperator: Yes, ma\'am. And would you like to add extra pecans in there?\\nCustomer: No, thank you.\\nOperator: What else can I get for you?\\nCustomer: That\'s it.\\nOperator: That\'ll be $5.40 for you.\\nCustomer: Okay, thank you.",\n    "complete_order": 1,\n    "mobile_order": 0,\n    "coupon_used": 0,\n    "asked_more_time": 0,\n    "out_of_stock_items": "0"\n}'

In [40]:
import json

data = json.loads(cleaned)

# Keep only what you want
filtered = {
    "text": data["text"],
}

print(filtered)


{'text': "Operator: It's a wonderful day at Dairy Queen. Will you be using your mobile rewards today?\nCustomer: No, thank you.\nOperator: What can I get for you?\nCustomer: Can I get a small pecan cluster blizzard, please?\nOperator: What flavor blizzard was that again?\nCustomer: Pecan cluster, turtle pecan cluster.\nOperator: Yes, ma'am. And would you like to add extra pecans in there?\nCustomer: No, thank you.\nOperator: What else can I get for you?\nCustomer: That's it.\nOperator: That'll be $5.40 for you.\nCustomer: Okay, thank you."}


In [41]:
#This is step1 transcript
filtered["text"]

"Operator: It's a wonderful day at Dairy Queen. Will you be using your mobile rewards today?\nCustomer: No, thank you.\nOperator: What can I get for you?\nCustomer: Can I get a small pecan cluster blizzard, please?\nOperator: What flavor blizzard was that again?\nCustomer: Pecan cluster, turtle pecan cluster.\nOperator: Yes, ma'am. And would you like to add extra pecans in there?\nCustomer: No, thank you.\nOperator: What else can I get for you?\nCustomer: That's it.\nOperator: That'll be $5.40 for you.\nCustomer: Okay, thank you."

In [42]:
import stable_whisper

In [43]:
model = stable_whisper.load_model("base")

In [44]:
result = model.align("Unclipped Processed Transactions/George_8.wav", transcription, language = "en")

Align: 100%|██████████| 25.81/25.81 [00:00<00:00, 43.32sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/alignment.py:211: UserWarning: Failed to align the last 2/77 words after 00:25.200.
  result = aligner.align(audio, text)
Adjustment: 100%|██████████| 25.812/25.812 [00:00<00:00, 31945.52sec/s]
/opt/anaconda3/envs/voice_id_env/lib/python3.11/site-packages/stable_whisper/alignment.py:211: UserWarning: 2/17 segments failed to align.
  result = aligner.align(audio, text)


In [45]:
import os
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"

In [46]:
result

In [47]:
result.to_srt_vtt("audio_2.srt")

Saved: /Users/anshulchiranth/Desktop/Strike/Voice Experiments/audio_2.srt


In [23]:
import os, sys, shutil, subprocess

print("python:", sys.executable)
print("ffmpeg (shutil.which):", shutil.which("ffmpeg"))
print("PATH:", os.environ.get("PATH","")[:300], "...")

python: /opt/anaconda3/envs/voice_id_env/bin/python
ffmpeg (shutil.which): /opt/homebrew/bin/ffmpeg
PATH: /opt/homebrew/bin:/opt/anaconda3/envs/voice_id_env/bin:/opt/anaconda3/bin:/opt/anaconda3/condabin:/usr/bin:/bin:/usr/sbin:/sbin ...
